In [ ]:
sys.path.insert(0, snakemake.input.data2dd)

In [ ]:
import numpy as np
import pandas as pd
import datetime
from data2dd_funcs import wrapdd

In [ ]:
year = snakemake.wildcards.year
dates = snakemake.params.date_range
dates = [year + "-" + date for date in dates]
dates = pd.date_range(dates[0], dates[1], freq='d')
dates

In [ ]:
summer_time_dates ={
    2010:[pd.Timestamp(2010,3,28), pd.Timestamp(2010,10,31)],
    2011:[pd.Timestamp(2011,3,27), pd.Timestamp(2011,10,30)],
    2012:[pd.Timestamp(2012,3,25), pd.Timestamp(2012,10,28)],
}

In [ ]:
UTC = pd.Series({
    'AL':1,
    'AT':1,
    'BE':1,
    'BA':1,
    'BG':2,
    'HR':1,
    'CZ':1,
    'CY':2,
    'DK':1,
    'EE':2,
    'FI':2,
    'FR':1,
    'DE':1,
    'GR':2,
    'HU':1,
    'IS':0,
    'IE':0,
    'IT':1,
    'XK':1,
    'LV':2,
    'LI':1,
    'LT':2,
    'LU':1,
    'MK':1,
    'MT':1,
    'ME':1,
    'NL':1,
    'NO':1,
    'PL':1,
    'PT':0,
    'RS':1,
    'RO':2,
    'SK':1,
    'SI':1,
    'ES':1,
    'SE':1,
    'CH':1,
    'UK':0,
})

In [ ]:
aggregated_regions = snakemake.params.aggregated_regions
aggregated_regions

In [ ]:
outdd = []

In [ ]:
# ev_scenarios.csv # contain the number of EVs and battery capacity per zone
ev_scenarios = snakemake.input["ev_scenarios"]

evs = pd.read_csv(ev_scenarios, index_col=(0,1)).loc[snakemake.params.ev_scenario[0], 'evs'][aggregated_regions].T.reset_index()
outdd.append(wrapdd(evs, "par_vehicles", "parameter"))

ecap = pd.read_csv(ev_scenarios, index_col=(0,1)).loc[snakemake.params.ev_scenario[0], 'ecap'][aggregated_regions].T.reset_index()
outdd.append(wrapdd(ecap, "par_ev_ecap", "parameter"))

In [ ]:
# EV_grid_connected_power_kW.csv

connected_vehicles = pd.read_csv(snakemake.input[0], index_col='date', parse_dates=True)[aggregated_regions]

connected = []

for date in dates:
    day = date.day_name()
    connected_vehicles_day = connected_vehicles.copy().query("index.dt.day_name() == @day")
    if date < summer_time_dates[int(year)][0] or date > summer_time_dates[int(year)][1]:
        for country in connected_vehicles_day.columns:
            connected_vehicles_day[country] = np.roll(connected_vehicles_day[country].values, UTC[country]*(-1))
    else:
        for country in connected_vehicles_day.columns:
            connected_vehicles_day[country] = np.roll(connected_vehicles_day[country].values, UTC[country]*(-1)-1)
    
    connected.append(connected_vehicles_day.set_index(pd.date_range(date.strftime('%Y-%m-%d'), date.strftime('%Y-%m-%d') + ' 23', freq='h')))

connected_vehicles = pd.concat(connected).reset_index(drop=True).melt(var_name='country', ignore_index=False).reset_index(names='time').set_index(['country','time']).div(1000).div(snakemake.params.ev_pcap).reset_index().astype({'time':str})
connected_vehicles['index'] = connected_vehicles['country'] + '.' + connected_vehicles['time']
connected_vehicles = connected_vehicles.drop(columns = ['country','time'])[['index','value']]
outdd.append(wrapdd(connected_vehicles, "par_grid_connected", "parameter"))

In [ ]:
# demand_driving_MWh.tsv

demand_driving = pd.read_csv(snakemake.input[1], index_col='date', parse_dates=True)[aggregated_regions]

demand = []

for date in dates:
    day = date.day_name()
    demand_driving_day = demand_driving.copy().query("index.dt.day_name() == @day")
    if date < summer_time_dates[int(year)][0] or date > summer_time_dates[int(year)][1]:
        for country in demand_driving_day.columns:
            demand_driving_day[country] = np.roll(demand_driving_day[country].values, UTC[country]*(-1))
    else:
        for country in demand_driving_day.columns:
            demand_driving_day[country] = np.roll(demand_driving_day[country].values, UTC[country]*(-1)-1)

    demand.append(demand_driving_day.set_index(pd.date_range(date.strftime('%Y-%m-%d'), date.strftime('%Y-%m-%d') + ' 23', freq='h')))

demand_driving = pd.concat(demand).reset_index(drop=True).melt(var_name='country', ignore_index=False).reset_index(names='time').set_index(['country','time']).div(1000).reset_index().astype({'time':str})
demand_driving['index'] = demand_driving['country'] + '.' + demand_driving['time'] + '.EV'
demand_driving = demand_driving.drop(columns = ['country','time'])[['index','value']]
outdd.append(wrapdd(demand_driving, "par_driving_demand", "parameter"))

In [ ]:
# demand_ev_charging_MWh.tsv

demand_charging = pd.read_csv(snakemake.input[2], index_col='date', parse_dates=True)[aggregated_regions]

charging = []

for date in dates:
    day = date.day_name()
    demand_charging_day = demand_charging.copy().query("index.dt.day_name() == @day")
    if date < summer_time_dates[int(year)][0] or date > summer_time_dates[int(year)][1]:
        for country in demand_charging_day.columns:
            demand_charging_day[country] = np.roll(demand_charging_day[country].values, UTC[country]*(-1))
    else:
        for country in demand_charging_day.columns:
            demand_charging_day[country] = np.roll(demand_charging_day[country].values, UTC[country]*(-1)-1)

    charging.append(demand_charging_day.set_index(pd.date_range(date.strftime('%Y-%m-%d'), date.strftime('%Y-%m-%d') + ' 23', freq='h')))

demand_charging = pd.concat(charging).reset_index(drop=True).melt(var_name='country', ignore_index=False).reset_index(names='time').set_index(['country','time']).div(1000).reset_index().astype({'time':str})
demand_charging['index'] = demand_charging['country'] + '.' + demand_charging['time']
demand_charging = demand_charging.drop(columns = ['country','time'])[['index','value']]
outdd.append(wrapdd(demand_charging, "par_ev_charging", "parameter"))

In [ ]:
outdd = np.concatenate(outdd, axis=0)

In [ ]:
np.savetxt(snakemake.output[0], outdd, delimiter=" ", fmt="%s")